In [1]:
import torch
from torch.autograd.functional import jacobian

# Trajectory and Energy Conservation

In [2]:
def trajectory_errors(x_true, x_pred, eps=1e-8):
    """
    Compute absolute and relative trajectory errors for q and p.

    Args:
        x_true: [T, 2n] ground truth states
        x_pred: [T, 2n] predicted states
        eps: small number to avoid divide-by-zero

    Returns:
        dict with abs/rel errors for q and p
    """
    n = x_true.shape[1] // 2
    q_true, p_true = x_true[:, :n], x_true[:, n:]
    q_pred, p_pred = x_pred[:, :n], x_pred[:, n:]

    abs_err_q = torch.norm(q_pred - q_true, dim=1)
    abs_err_p = torch.norm(p_pred - p_true, dim=1)

    rel_err_q = abs_err_q / (torch.norm(q_true, dim=1) + eps)
    rel_err_p = abs_err_p / (torch.norm(p_true, dim=1) + eps)

    return {
        "abs_err_q": abs_err_q,
        "abs_err_p": abs_err_p,
        "rel_err_q": rel_err_q,
        "rel_err_p": rel_err_p,
        "mean_abs_err_q": abs_err_q.mean(),
        "mean_abs_err_p": abs_err_p.mean(),
        "mean_rel_err_q": rel_err_q.mean(),
        "mean_rel_err_p": rel_err_p.mean()
    }


def energy_errors(H_func, x_traj):
    """
    Compute energy conservation errors along a trajectory.

    Args:
        H_func: callable H(x) -> scalar
        x_traj: [T, 2n] trajectory

    Returns:
        dict with absolute and relative energy errors
    """
    H0 = H_func(x_traj[0])
    H_vals = torch.tensor([H_func(x) for x in x_traj])

    abs_err = torch.abs(H_vals - H0)
    rel_err = abs_err / (torch.abs(H0) + 1e-8)

    return {
        "abs_err": abs_err,
        "rel_err": rel_err,
        "max_abs_err": abs_err.max(),
        "max_rel_err": rel_err.max(),
        "mean_abs_err": abs_err.mean(),
        "mean_rel_err": rel_err.mean()
    }

In [3]:
# Example

T = 5           # number of time steps
n = 1           # number of degrees of freedom
dt = 0.1        # time step size

# Ground truth trajectory: 1D harmonic oscillator (q,p)
# q(t) = cos(t), p(t) = -sin(t)
times = torch.arange(0, T*dt, dt)
q_true = torch.cos(times).unsqueeze(1)        # shape [T,1]
p_true = -torch.sin(times).unsqueeze(1)       # shape [T,1]
x_true = torch.cat([q_true, p_true], dim=1)   # shape [T, 2n]

# Predicted trajectory with small noise
x_pred = x_true + 0.01 * torch.randn_like(x_true)

# Hamiltonian function for harmonic oscillator: H = 0.5*(p^2 + q^2)
def H_func(x):
    q, p = x[0], x[1]
    return 0.5 * (q**2 + p**2)

# -----------------------------
# Run trajectory error test
# -----------------------------
traj_errs = trajectory_errors(x_true, x_pred)
print("Trajectory Errors:")
for k, v in traj_errs.items():
    if "mean" not in k:  # print first few values
        print(f"{k}: {v}")

print("\nMean Errors:")
print(f"mean_abs_err_q: {traj_errs['mean_abs_err_q'].item():.5f}")
print(f"mean_abs_err_p: {traj_errs['mean_abs_err_p'].item():.5f}")
print(f"mean_rel_err_q: {traj_errs['mean_rel_err_q'].item():.5f}")
print(f"mean_rel_err_p: {traj_errs['mean_rel_err_p'].item():.5f}")

# -----------------------------
# Run energy error test
# -----------------------------
energy_errs = energy_errors(H_func, x_pred)
print("\nEnergy Errors:")
print(f"abs_err: {energy_errs['abs_err']}")
print(f"rel_err: {energy_errs['rel_err']}")
print(f"max_abs_err: {energy_errs['max_abs_err'].item():.5f}")
print(f"max_rel_err: {energy_errs['max_rel_err'].item():.5f}")
print(f"mean_abs_err: {energy_errs['mean_abs_err'].item():.5f}")
print(f"mean_rel_err: {energy_errs['mean_rel_err'].item():.5f}")


Trajectory Errors:
abs_err_q: tensor([0.0105, 0.0043, 0.0037, 0.0036, 0.0038])
abs_err_p: tensor([0.0073, 0.0253, 0.0010, 0.0060, 0.0002])
rel_err_q: tensor([0.0105, 0.0044, 0.0037, 0.0037, 0.0041])
rel_err_p: tensor([7.2638e+05, 2.5300e-01, 5.0430e-03, 2.0188e-02, 5.8010e-04])

Mean Errors:
mean_abs_err_q: 0.00517
mean_abs_err_p: 0.00794
mean_rel_err_q: 0.00528
mean_rel_err_p: 145276.03125

Energy Errors:
abs_err: tensor([0.0000, 0.0089, 0.0066, 0.0052, 0.0070])
rel_err: tensor([0.0000, 0.0182, 0.0135, 0.0107, 0.0142])
max_abs_err: 0.00891
max_rel_err: 0.01820
mean_abs_err: 0.00554
mean_rel_err: 0.01131


# Autoencoder Reconstruction and Symplecticity

In [4]:
def reconstruction_error(x, x_hat, eps=1e-8):
    """
    Compute absolute and relative reconstruction error for an autoencoder.

    Args:
        x: [T, 2n] original states
        x_hat: [T, 2n] reconstructed states
        eps: small number to avoid divide-by-zero

    Returns:
        dict with abs and relative errors
    """
    abs_err = torch.norm(x - x_hat, dim=1)
    rel_err = abs_err / (torch.norm(x, dim=1) + eps)

    return {
        "abs_err": abs_err,
        "rel_err": rel_err,
        "mean_abs_err": abs_err.mean(),
        "mean_rel_err": rel_err.mean(),
        "max_abs_err": abs_err.max(),
        "max_rel_err": rel_err.max()
    }


def ae_symplecticity_error(autoencoder_map, x_traj):
    """
    Check approximate symplecticity of the autoencoder mapping x -> x_hat
    Args:
        autoencoder_map: callable x -> x_hat (encoder + decoder)
        x_traj: [T, 2n] trajectory of original states
    Returns:
        dict with max and mean symplectic deviation across trajectory
    """
    n = x_traj.shape[1] // 2
    device = x_traj.device
    dtype = x_traj.dtype

    # Symplectic form Ω
    Omega = torch.zeros(2*n, 2*n, dtype=dtype, device=device)
    Omega[:n, n:] = torch.eye(n, dtype=dtype, device=device)
    Omega[n:, :n] = -torch.eye(n, dtype=dtype, device=device)

    symp_errors = []

    for x in x_traj:
        x = x.detach().requires_grad_(True)
        x_flat = x.view(-1)  # ensure 1D
        J = jacobian(autoencoder_map, x_flat)  # [2n,2n]
        symp_err = (J.T @ Omega @ J - Omega).abs().max().item()
        symp_errors.append(symp_err)

    symp_errors = torch.tensor(symp_errors)
    return {
        "max_symplectic_error": symp_errors.max().item(),
        "mean_symplectic_error": symp_errors.mean().item()
    }


In [5]:
# -----------------------------
# Dummy Encoder and Decoder
# -----------------------------
class DummyEncoder(torch.nn.Module):
    def __init__(self, noise=0.01):
        super().__init__()
        self.noise = noise

    def forward(self, x):
        # simple linear map + small noise to simulate encoding
        return x * 0.5 + self.noise * torch.randn_like(x)

class DummyDecoder(torch.nn.Module):
    def __init__(self, noise=0.01):
        super().__init__()
        self.noise = noise

    def forward(self, z):
        # simple linear map back to original space + small noise
        return z * 2.0 + self.noise * torch.randn_like(z)

# -----------------------------
# Dummy trajectory: 1D harmonic oscillator
# -----------------------------
T = 5
n = 1
dt = 0.1
times = torch.arange(0, T*dt, dt)
q = torch.cos(times).unsqueeze(1)      # [T,1]
p = -torch.sin(times).unsqueeze(1)     # [T,1]
x_traj = torch.cat([q, p], dim=1)      # [T,2n]

# -----------------------------
# Initialize encoder + decoder
# -----------------------------
encoder = DummyEncoder(noise=5)
decoder = DummyDecoder(noise=0.10)

# -----------------------------
# Reconstruction error
# -----------------------------
x_hat = decoder(encoder(x_traj))
rec_errs = reconstruction_error(x_traj, x_hat)
print("Reconstruction Errors:")
print(f"Mean absolute error: {rec_errs['mean_abs_err']:.5f}")
print(f"Mean relative error: {rec_errs['mean_rel_err']:.5f}")
print(f"Max absolute error: {rec_errs['max_abs_err']:.5f}")
print(f"Max relative error: {rec_errs['max_rel_err']:.5f}")

# -----------------------------
# Symplecticity error
# -----------------------------
symp_errs = ae_symplecticity_error(encoder, x_traj)
print("\nSymplecticity Errors:")
print(f"Mean symplectic error: {symp_errs['mean_symplectic_error']:.5f}")
print(f"Max symplectic error: {symp_errs['max_symplectic_error']:.5f}")

# we should find 0.75 (1 - 0.5 * 0.5)

Reconstruction Errors:
Mean absolute error: 12.42488
Mean relative error: 12.42488
Max absolute error: 19.06516
Max relative error: 19.06516

Symplecticity Errors:
Mean symplectic error: 0.75000
Max symplectic error: 0.75000
